<a href="https://colab.research.google.com/github/Imrane206/-Classification-via-CNN-ResNet_b0-/blob/main/AI_Financial_Report_Summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Install Unsloth, Xformers (for memory efficient attention) and other dependencies


In [1]:
# Install Unsloth, Xformers (for memory efficient attention) and other dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-if1l6e_f/unsloth_b506a21782374060b7e1044aba596fad
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-if1l6e_f/unsloth_b506a21782374060b7e1044aba596fad
  Resolved https://github.com/unslothai/unsloth.git to commit 0326577b821603878ae9d85b67d52eb09bf7ae46
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.8.6
    Uninstalling trl-0.8.6:
      Successfully uninstalled trl-0.8.6
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully 

# Loading LLAMA 3.1 8B The base Model with Quatization 4_bit

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Supports RoPE Scaling internally for longer sequences
dtype = None # None for auto detection. Float16 for T4, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage by 4x

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3.1-8b-instruct-bnb-4bit", # Highly optimized 4-bit version
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)
# 2. 🔥 IMPORTANT: Fix tokenizer
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.


# Applying LORA Configuration on LLAMA 3.1 8B



In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: 8, 16, 32, 64 are common. 16 is a great balance for summarization.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0.05, # Optimized to 0 for Unsloth speed
    bias = "none",    # Optimized to "none" for Unsloth
    use_gradient_checkpointing = "unsloth", # Saves massive VRAM
    random_state = 3407,
    use_rslora = False,  # Rank Stabilized LoRA
    loftq_config = None, # Not needed for initial fine-tuning
)


print("✅ Model and LoRA adapters ready for training!")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.8 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ Model and LoRA adapters ready for training!


# Loading the Dataset From Huggingface

In [4]:
from datasets import load_dataset

# 1. Load and Shuffle
print("Loading FINDSum...")
dataset = load_dataset("datht/FINDSum", split="train")
print(dataset.features)



Loading FINDSum...


README.md:   0%|          | 0.00/122 [00:00<?, ?B/s]

FINDSum/train.csv:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

FINDSum/validation.csv:   0%|          | 0.00/143M [00:00<?, ?B/s]

FINDSum/test.csv:   0%|          | 0.00/143M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/83255 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10406 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10406 [00:00<?, ? examples/s]

{'document': Value('string'), 'summary': Value('string')}


# Using a small sample from the Dataset $$$ Data preprocessing

In [5]:
# 2. Filter out short/empty rows (prevents training on noise)
from datasets import load_dataset

# 1. Load dataset
dataset = load_dataset("datht/FINDSum", split="train")

# 2. Shuffle + reduce size (for T4)
dataset = dataset.shuffle(seed=42).select(range(500))

# 3. Filter useful samples
dataset = dataset.filter(
    lambda x: len(x['document']) > 200 and len(x['summary']) > 20
)

# 4. Prompt formatting (FIXED: Word-based Truncation)
def format_prompt(example):
    instructions = "Analyze the following financial report section and provide a concise, accurate summary."

    # 🚨 FIX: Truncate by words to prevent cutting numbers (e.g., "$10,") in half.
    # A standard English word is ~1.3 tokens.
    # 1200 words = ~1500 tokens, leaving plenty of room for the summary in a 2048 context window.
    document_words = str(example['document']).split()
    safe_document = " ".join(document_words[:1200])

    summary_words = str(example['summary']).split()
    safe_summary = " ".join(summary_words[:150])

    formatted_text = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        f"{instructions}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{safe_document}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{safe_summary}<|eot_id|>"
    )

    return {"text": formatted_text}

# 5. Apply formatting + REMOVE UNUSED COLUMNS ✅
processed_dataset = dataset.map(
    format_prompt,
    remove_columns=dataset.column_names
)

# 6. Correct print ✅
print(f"✅ Data ready. Sample length: {len(processed_dataset[0]['text'])} chars.")

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

✅ Data ready. Sample length: 8129 chars.


# Training Configurations $$$ Tokenization

In [6]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
import transformers.trainer_utils
from trl import DataCollatorForCompletionOnlyLM # <-- 1. Import the new collator

# Monkey-patch the quantization validation to skip the check for LoRA models
original_validate = transformers.trainer_utils.validate_quantization_for_training

def patched_validate(model):
    # If model has LoRA adapters (is a PeftModel), skip the check
    if hasattr(model, 'peft_config') or hasattr(model, 'base_model'):
        return
    original_validate(model)

transformers.trainer_utils.validate_quantization_for_training = patched_validate

# Tokenize the dataset
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        max_length=max_seq_length,
        return_tensors=None
    )
    # <-- 2. Removed `model_inputs["labels"] = ...` The collator handles this dynamically now.
    return model_inputs

dataset_dict = processed_dataset.train_test_split(test_size=0.1, seed=42)

tokenized_train = dataset_dict["train"].map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_eval = dataset_dict["test"].map(tokenize_function, batched=True, remove_columns=["text"])


# <-- 3. INTEGRATE LABEL MASKING HERE -->
# Define the exact token sequence that immediately precedes the assistant's response
response_template = "<|start_header_id|>assistant<|end_header_id|>\n\n"
response_template_ids = tokenizer.encode(response_template, add_special_tokens=False)

# This collator finds the response template and masks everything before it with -100
data_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,
    tokenizer=tokenizer
)


training_args = TrainingArguments(
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs =2,
    learning_rate=2e-4,
    warmup_steps=50,
    fp16=True,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    logging_steps=25,
    logging_first_step=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_steps=100,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    output_dir="outputs",
    seed=3407,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator, # <-- Now using the masked collator
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("✅ Trainer initialized successfully with label masking!")
print("🚀 Ready to train with: trainer.train()")

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

✅ Trainer initialized successfully with label masking!
🚀 Ready to train with: trainer.train()


# Training setup

In [7]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 2 | Total steps = 114
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,1.567319,1.490041
2,1.477652,1.463753


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=114, training_loss=1.5388605448237636, metrics={'train_runtime': 5221.1056, 'train_samples_per_second': 0.172, 'train_steps_per_second': 0.022, 'total_flos': 6.899280049874534e+16, 'train_loss': 1.5388605448237636, 'epoch': 2.0})

# Saving The model

In [8]:
import os

# Define your save path (e.g., inside your Google Drive)
save_directory = "/content/drive/MyDrive/Financial_Summarizer_LoRA"

# 1. Save ONLY the LoRA adapters and config
model.save_pretrained(save_directory)

# 2. Save the tokenizer so you can format inputs correctly later
tokenizer.save_pretrained(save_directory)

print(f"✅ LoRA weights successfully saved to {save_directory}")
# You should notice this takes up less than ~300 MB!

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Financial_Summarizer_LoRA/tokenizer_config.json.


✅ LoRA weights successfully saved to /content/drive/MyDrive/Financial_Summarizer_LoRA


# Evaluation Step

In [9]:
# Install evaluation libraries
!pip install rouge_score nltk bert-score -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.7 MB/s eta 0:00:00


In [10]:
from rouge_score import rouge_scorer
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
import torch

def evaluate_model_on_findsum(
    model,
    tokenizer,
    num_samples=50,
    show_examples=5
):
    """
    Complete evaluation on FINDSum dataset
    """
    print("="*80)
    print("📊 FINANCIAL SUMMARIZATION MODEL EVALUATION")
    print("="*80)

    # Step 1: Load test dataset
    # -------------------------------------------------------------------------
    print("\n[Step 1/4] Loading and preparing test dataset...")
    # FIX: Changed EVA_DATA to dataset so the next lines reference it correctly
    dataset = load_dataset("datht/FINDSum", split="train")
    dataset = dataset.shuffle(seed=42)

    # Use different samples than training to prevent data leakage (offset by 500)
    test_dataset = dataset.select(range(500, min(500 + num_samples, len(dataset))))

    # Filter
    test_dataset = test_dataset.filter(
        lambda x: len(x['document']) > 200 and len(x['summary']) > 20
    )

    print(f"✓ Loaded {len(test_dataset)} test samples")

    # Step 2: Generate summaries
    # -------------------------------------------------------------------------
    print("\n[Step 2/4] Generating summaries...")

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    examples_to_show = []

    model.eval()  # Set to eval mode

    for i in tqdm(range(len(test_dataset))):
        example = test_dataset[i]

        # Word-based truncation to match training
        document_words = str(example['document']).split()
        document = " ".join(document_words[:1200])

        reference_words = str(example['summary']).split()
        reference = " ".join(reference_words[:150])

        # Generate summary
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Analyze the following financial report section and provide a concise, accurate summary.<|eot_id|><|start_header_id|>user<|end_header_id|>

{document}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1
            )

        generated = tokenizer.decode(outputs[0], skip_special_tokens=False)

        # Extract summary securely
        split_token = "<|start_header_id|>assistant<|end_header_id|>\n\n"
        if split_token in generated:
            generated_summary = generated.split(split_token)[-1]
        else:
            generated_summary = generated

        # Clean up trailing tokens
        generated_summary = generated_summary.split("<|eot_id|>")[0].strip()
        generated_summary = generated_summary.replace("<|begin_of_text|>", "").strip()

        # Calculate ROUGE scores
        scores = scorer.score(reference, generated_summary)
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)

    # Step 3: Calculate statistics
    # -------------------------------------------------------------------------
    print("\n[Step 3/4] Calculating statistics...")

    results = {
        'ROUGE-1': {
            'mean': np.mean(rouge1_scores),
            'median': np.median(rouge1_scores),
        },
        'ROUGE-2': {
            'mean': np.mean(rouge2_scores),
            'median': np.median(rouge2_scores),
        },
        'ROUGE-L': {
            'mean': np.mean(rougeL_scores),
            'median': np.median(rougeL_scores),
        }
    }

    print("\n✅ Evaluation Complete!")
    print(results)
    return results

# Run the function to actually start the evaluation!
eval_results = evaluate_model_on_findsum(model, tokenizer)

📊 FINANCIAL SUMMARIZATION MODEL EVALUATION

[Step 1/4] Loading and preparing test dataset...


Filter:   0%|          | 0/50 [00:00<?, ? examples/s]

✓ Loaded 50 test samples

[Step 2/4] Generating summaries...


  0%|          | 0/50 [00:00<?, ?it/s]Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, F


[Step 3/4] Calculating statistics...

✅ Evaluation Complete!
{'ROUGE-1': {'mean': np.float64(0.26981286889080036), 'median': np.float64(0.24963614373859744)}, 'ROUGE-2': {'mean': np.float64(0.059461446418731266), 'median': np.float64(0.03484362859362859)}, 'ROUGE-L': {'mean': np.float64(0.15907197183253113), 'median': np.float64(0.14902348035988588)}}


# Inference Part (Summary Generation)  ,,, On teste la performance de notre model

In [2]:
import torch
from unsloth import FastLanguageModel

# ==========================================
# 1. CONFIGURATION & LOADING
# ==========================================
# Point this directly to the folder in your Google Drive
save_directory = "/content/drive/MyDrive/Financial_Summarizer_LoRA"
max_seq_length = 2048

print("🚀 Initializing inference environment...")

# Unsloth reads the adapter_config.json here, automatically pulls the base
# Llama 3.1 8B model into memory, and then seamlessly attaches your LoRA weights.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = save_directory,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True, # Keeps memory footprint low so it runs easily on a Colab T4
)

# ==========================================
# 2. OPTIMIZATION
# ==========================================
# This is a key feature to mention during your presentation!
# It enables Unsloth's native optimized inference for 2x faster text generation.
FastLanguageModel.for_inference(model)

# ==========================================
# 3. PREPARE THE TEST DATA
# ==========================================
test_document = """
intangible assets - we allocate a portion of the purchase price of acquisitions to identifiable intangible assets and we amortize definite-lived assets over their estimated useful lives . we consider our indefinite-lived assets for impairment whenever events or changes in circumstances indicate that the carrying amount of an asset may not be recoverable . trade names are not amortized as they are believed to have an indefinite life . trade names are reviewed annually for impairment under asc 350. as a result of our acquisition of vertro , inc. ( `` vertro `` ) in march 2012 , we recognized an asset for the customer relationship with google of $ 8,820,000 and assigned it a useful life of 20 years . a primary reason for acquiring vertro was its relationship with google . up to the time of the acquisition , we principally had access to the yahoo ! inventory of advertisements . among the many valuable assets acquired in the vertro transaction was this google relationship and the access it provided to an enormous inventory of advertisements . in addition , we acquired the alot brand , whose products are monetized through google and has historically produced a better margin than monetization through yahoo ! . in determining the useful life of this asset , we considered the strategic importance of vertro 's strong relationship with google . vertro and its predecessor company had contracts and successful renewals with google that date back to 2006. the google contract has been extended through february 28 , 2019. we expect the relationship with google to continue through the 20 -year amortization period and beyond . at the time of the vertro acquisition , we engaged a third party valuation service to determine the fair value of the acquired assets . at the close of the 2017 and 2016 fiscal years , we again engaged a third party valuation service to reassess the fair value of the acquired assets . from time to time , both search marketplaces , google and yahoo ! , may implement policy or marketplace changes . in january 2013 google requested changes to our agreement that impacted marketing programs for one of our alot products , the appbar , the result of which was a decline in the number of product installs . since acquiring the alot brand in the vertro acquisition , we have materially expanded the brand into a number of additional owned and operated websites and applications . we expect products within the brand to ebb and flow as customer preferences change and google adjusts its marketplace policies . at the close of 2013 , we considered the google change and decided to transition out of the appbar product and replace it with web properties that we develop . at the close of 2014 , we determined that the asset continued to be recoverable despite the impact to the appbar product and our decision to transition away from it . we made this determination in part because during 2014 we completely replaced the revenue and margin from the appbar product with other alot-branded and google monetized products . in may 2015 , we purchased two domain websites and recorded the purchase at $ 715,874 . in 2017 , we determined that the seller would not meet the specific performance target for the second and third years and therefore , we adjusted the carrying value of the intangible asset to $ 300,000 . we recorded no impairment of intangible assets during 2017 or 2016 . see note 5 , intangible assets and goodwill , for more information . income taxes - we utilize the liability method of accounting for income taxes as set forth in asc 740 , income taxes ( “ asc 740 ” ) . under the liability method , deferred taxes are determined based on the temporary differences between the financial statement and tax bases of assets and liabilities . a valuation allowance is recorded when it is more likely than not that some of the deferred tax assets will not be realized . in assessing the need for a valuation allowance , we must project future levels of taxable income , which requires significant judgment . we examine evidence related to the history of taxable losses or income , the economic conditions in which we operate , organizational characteristics , our forecasts and projections , as well as factors affecting liquidity . all our deferred tax assets and liabilities are recorded as long-term assets and liabilities in the consolidated balance sheets . in 2017 , we recognized an income tax benefit of $ 1,498,076 due to the passing of the tax cuts and jobs act in december 2017. the new law reduced corporate income tax rates from 35 % to 21 % . as a result , the deferred tax assets and liabilities recorded in the consolidated balance sheet were reevaluated at the new tax rates . both the deferred tax assets and the deferred tax liabilities were reduced . the decrease in the deferred tax liability resulted in the one-time tax benefit . we believe it is more likely than not that essentially none of our deferred tax assets will be realized , and we have recorded a full valuation for the net deferred tax assets as of december 31 , 2017 and 2016 . we have adopted certain provisions of asc 740. this statement clarifies the criteria that an individual tax position must satisfy for some or all of the benefits of that position to be recognized in a company 's financial statements . asc 740 prescribes a f-9 recognition threshold of more likely than not , and a measurement attribute for all tax positions taken or expected to be taken on a tax return , in order to be recognized in the financial statements . story_separator_special_tag continued the migration to mobile , going from 52.4 % mobile in 2016 to 62.1 % in 2017. critical accounting policies and estimates the preparation of financial statements in conformity with u.s. gaap requires management to make estimates and assumptions that affect the reported amount of assets and liabilities , the disclosure of contingent assets and liabilities and the reported amounts of revenue and expenses during the reported periods . the more critical accounting estimates include estimates related to revenue recognition and accounts receivable allowances . we also have other key accounting policies , which involve the use of estimates , judgments and assumptions that are significant to understanding our results , which are described in note 2 to our audited financial statements for 2017 and 2016 appearing elsewhere in this report . results of operations replace_table_token_2_th net revenue net revenue for the year ended december 31 , 2017 was $ 79.6 million compared to $ 71.5 million for the year ended december 31 , 2016 . the increase was primarily due to growth in the business acquired in february 2017. revenue from the acquired business line grew from $ 0.8 million in its first month of operations , february 2017 to $ 1.9 million in december 2017. we expect the new business line to continue to fuel company growth into the future . revenue from our validclick business , serving advertisements to publisher partners , increased 25 % in 2017 compared to 2016. revenue from our digital publishing business declined as we redirected resources and investment to the acquired business which has higher gross margins . the fourth quarter is traditionally the highest revenue quarter of the year . in 2017 , the fourth quarter revenue was $ 23.8 million , 21 % greater than the same quarter in 2016. the higher revenue in this year 's fourth quarter is attributable primarily to the new business line . cost of revenue cost of revenue is primarily generated by payments to website and application publishers who host our advertisements . the increase in cost of revenue in 2017 compared to 2016 is due to with the higher revenue from the acquired business and validclick . operating expenses replace_table_token_3_th 14 operating expenses decreased in the twelve months ended december 31 , 2017 as compared to the same period of the prior year . marketing costs or tac include those expenses required to attract traffic to our owned web properties . the decrease in marketing costs in the twelve months ended december 31 , 2017 was a strategy initiated at the beginning of 2017 in anticipation of the acquisition in february 2017. this strategy was designed to focus resources and investment towards a higher growth and gross margin ( after tac ) business at the expense of growth in another business line at lower gross margin . compensation expense increased 49.3 % in the twelve months ended december 31 , 2017 due primarily to an increase in the number of employees . the higher headcount is primarily due to the additional employees from the february 2017 acquisition . our total employment , both full-time and part-time , was 89 at december 31 , 2017 compared to 72 at the same time last year . we expect compensation expense to increase , though moderately , in the coming quarters as we hire additional developers and sales personnel to support the anticipated growth . selling , general and administrative costs were $ 8.3 million , an increase of 67.0 % over 2016. the primary reasons for the higher cost in the twelve months ended december 31 , 2017 compared to the same period last year is due to the acquisition in february 2017. among the higher 2017 expenses compared to 2016 were it costs approximately $ 1.3 million higher ; amortization and depreciation expense approximately $ 852,000 higher ; facilities cost approximately $ 290,000 higher and travel and entertainment costs approximately $ 255,000 higher . we expect selling , general and administrative costs to decrease in 2018. interest expense , net interest expense , net , which represents interest expense on the bank credit facility , was higher in 2017 compared to the same periods in 2016 because of a higher average outstanding revolving credit line balance and higher interest rates this year compared to last year . income tax benefit in 2017 , we recognized an income tax benefit of $ 1,498,076 due to the passing of the tax cuts and jobs act in december 2017. the new law reduced corporate income tax rates from 35 % to 21 % . as a result , the deferred tax assets and liabilities recorded in the consolidated balance sheet were reevaluated at the new tax rates . both the deferred tax assets and the deferred tax liabilities were reduced . the decrease in the deferred tax liability resulted in the one-time tax benefit . in 2016 , we recognized an income tax benefit of $ 29,260 . income ( loss ) from discontinued operations certain of our subsidiaries previously operated in the european union ( `` eu `` ) . though operations ceased in 2009 , statutory requirements made it necessary to have a continued presence in the eu for varying terms until november 2015. profits and losses generated from the remaining assets and liabilities are accounted for as discontinued operations . in the third quarter of 2016 , our petition with the uk ( united kingdom ) companies house to strike off and dissolve the remaining subsidiary in the eu was approved . as a result , for the twelve months ended december 31 , 2017 , we recognized a net loss from discontinued operations of $ 1,109 due to a charge from a service provider . as of december 31 , 2016 , we recorded a net income of $ 155,287 due primarily to the adjustment of certain accrued liabilities . no further charges or adjustments are expected . story_separator_special_tag style= `` line-height:120
"""

# Format the prompt EXACTLY as it was formatted during training, including special tokens.
instructions = "Analyze the following financial report section and provide a concise, accurate summary."

prompt = (
    "<|begin_of_text|>"
    "<|start_header_id|>system<|end_header_id|>\n\n"
    f"{instructions}<|eot_id|>"
    "<|start_header_id|>user<|end_header_id|>\n\n"
    f"{test_document.strip()}<|eot_id|>"
    "<|start_header_id|>assistant<|end_header_id|>\n\n"
)

# ==========================================
# 4. GENERATE SUMMARY
# ==========================================
print("⚙️ Tokenizing input and generating summary...")

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=max_seq_length
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,    # Cap the summary length
        do_sample=False,       # Strict factual decoding; prevents hallucinations with numbers
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.1 # Prevents the model from repeating phrases
    )

# ==========================================
# 5. FORMAT AND DISPLAY OUTPUT
# ==========================================
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Clean up the output to extract only the assistant's response
if "assistant" in generated_text:
    summary = generated_text.split("assistant")[-1].strip()
else:
    summary = generated_text

print("\n" + "="*70)
print("📄 SOURCE DOCUMENT:")
print("-" * 70)
print(test_document.strip())
print("\n" + "="*70)
print("📊 AI GENERATED SUMMARY:")
print("-" * 70)
print(summary)
print("="*70)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🚀 Initializing inference environment...
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.8 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


⚙️ Tokenizing input and generating summary...


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1


📄 SOURCE DOCUMENT:
----------------------------------------------------------------------
intangible assets - we allocate a portion of the purchase price of acquisitions to identifiable intangible assets and we amortize definite-lived assets over their estimated useful lives . we consider our indefinite-lived assets for impairment whenever events or changes in circumstances indicate that the carrying amount of an asset may not be recoverable . trade names are not amortized as they are believed to have an indefinite life . trade names are reviewed annually for impairment under asc 350. as a result of our acquisition of vertro , inc. ( `` vertro `` ) in march 2012 , we recognized an asset for the customer relationship with google of $ 8,820,000 and assigned it a useful life of 20 years . a primary reason for acquiring vertro was its relationship with google . up to the time of the acquisition , we principally had access to the yahoo ! inventory of advertisements . among the many valuabl